In [ ]:
# Install missing frameworks
!pip install -q lightning flax onnx onnxruntime tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 22.8 MB/s eta 0:00:00


In [ ]:
# Setup Models
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import tensorflow as tf
from tensorflow.keras import layers, models
import jax
import jax.numpy as jnp
from flax import linen as flax_nn
import onnxruntime as ort

print("PyTorch Version:", torch.__version__)
print("TensorFlow Version:", tf.__version__)
print("JAX Version:", jax.__version__)

PyTorch Version: 2.11.0+cu128
TensorFlow Version: 2.20.0
JAX Version: 0.7.2


In [ ]:
# --- 1. PyTorch Model ---
class PyTorchEMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 47)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model_pt = PyTorchEMNIST().eval()

In [ ]:
# --- 2. PyTorch Lightning Model ---
class LitEMNIST(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = PyTorchEMNIST()

    def forward(self, x):
        return self.model(x)

model_lit = LitEMNIST().eval()

In [ ]:
# --- 3. TensorFlow / Keras Model ---
model_tf = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(47)
])

In [ ]:
# --- 4. JAX / Flax Model ---
class FlaxEMNIST(flax_nn.Module):
    @flax_nn.compact
    def __call__(self, x):
        x = flax_nn.Conv(features=32, kernel_size=(3, 3))(x)
        x = flax_nn.relu(x)
        x = flax_nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))
        x = flax_nn.Conv(features=64, kernel_size=(3, 3))(x)
        x = flax_nn.relu(x)
        x = flax_nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))
        x = x.reshape((x.shape[0], -1))
        x = flax_nn.Dense(features=128)(x)
        x = flax_nn.relu(x)
        x = flax_nn.Dense(features=47)(x)
        return x

flax_model = FlaxEMNIST()
rng = jax.random.PRNGKey(42)
dummy_jax_input = jnp.ones((1, 28, 28, 1))
flax_params = flax_model.init(rng, dummy_jax_input)

# JIT-compile inference function for high performance
@jax.jit
def jax_predict(params, x):
    return flax_model.apply(params, x)

In [ ]:
!pip install -q onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 10.1 MB/s eta 0:00:00


In [ ]:
# --- 5. ONNX Runtime (Export from PyTorch) ---
dummy_onnx_input = torch.randn(1, 1, 28, 28)
onnx_path = "emnist_model.onnx"

torch.onnx.export(
    model_pt,
    dummy_onnx_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False  # Skips onnxscript dependency
)

ort_session = ort.InferenceSession(onnx_path)
ort_input_name = ort_session.get_inputs()[0].name

/tmp/ipykernel_762/3480660889.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [ ]:
# Benchmark Runner
def benchmark_inference(predict_fn, sample_input, num_warmup=50, num_runs=500):
    # Warmup phase
    for _ in range(num_warmup):
        _ = predict_fn(sample_input)

    # Timed benchmark phase
    start_time = time.perf_counter()
    for _ in range(num_runs):
        _ = predict_fn(sample_input)
    end_time = time.perf_counter()

    avg_latency_ms = ((end_time - start_time) / num_runs) * 1000.0
    return avg_latency_ms

In [ ]:
# Cell 4: Run Tests & Build Pandas Comparison Table
results = []
batch_sizes = [1, 64]

for b in batch_sizes:
    # Prepare inputs according to framework shape conventions
    pt_input = torch.randn(b, 1, 28, 28)           # (B, C, H, W)
    tf_input = tf.random.normal((b, 28, 28, 1))     # (B, H, W, C)
    jax_input = jnp.ones((b, 28, 28, 1))           # (B, H, W, C)
    ort_input = np.random.randn(b, 1, 28, 28).astype(np.float32)

    # 1. PyTorch
    with torch.no_grad():
        t_pt = benchmark_inference(lambda x: model_pt(x), pt_input)

    # 2. PyTorch Lightning
    with torch.no_grad():
        t_lit = benchmark_inference(lambda x: model_lit(x), pt_input)

    # 3. TensorFlow / Keras
    t_tf = benchmark_inference(lambda x: model_tf(x, training=False), tf_input)

    # 4. JAX / Flax (block until computation is fully finished on device)
    t_jax = benchmark_inference(lambda x: jax_predict(flax_params, x).block_until_ready(), jax_input)

    # 5. ONNX Runtime
    t_ort = benchmark_inference(lambda x: ort_session.run(None, {ort_input_name: x}), ort_input)

    results.append({
        "Batch Size": b,
        "PyTorch (ms)": round(t_pt, 3),
        "PyTorch Lightning (ms)": round(t_lit, 3),
        "TensorFlow/Keras (ms)": round(t_tf, 3),
        "JAX / Flax (JIT) (ms)": round(t_jax, 3),
        "ONNX Runtime (ms)": round(t_ort, 3)
    })

# Convert to clean Comparison DataFrame
df_raw = pd.DataFrame(results)

# Transpose for easier reading by Framework
summary = []
frameworks = [
    ("PyTorch", "PyTorch (ms)"),
    ("PyTorch Lightning", "PyTorch Lightning (ms)"),
    ("TensorFlow / Keras", "TensorFlow/Keras (ms)"),
    ("JAX / Flax (JIT)", "JAX / Flax (JIT) (ms)"),
    ("ONNX Runtime", "ONNX Runtime (ms)")
]

for name, col in frameworks:
    lat_b1 = df_raw.loc[df_raw["Batch Size"] == 1, col].values[0]
    lat_b64 = df_raw.loc[df_raw["Batch Size"] == 64, col].values[0]
    throughput_b64 = int((64 / (lat_b64 / 1000.0)))

    summary.append({
        "Framework": name,
        "Latency B=1 (ms)": f"{lat_b1:.3f} ms",
        "Latency B=64 (ms)": f"{lat_b64:.3f} ms",
        "Throughput (img/sec)": f"{throughput_b64:,} items/s",
        "Primary Advantage": {
            "PyTorch": "Ecosystem & dynamic graph",
            "PyTorch Lightning": "Clean production training code",
            "TensorFlow / Keras": "Beginner-friendly API",
            "JAX / Flax (JIT)": "XLA-compiled functional speed",
            "ONNX Runtime": "Graph-optimized CPU inference"
        }[name]
    })

df_summary = pd.DataFrame(summary).sort_values(by="Latency B=1 (ms)")

print("=" * 70)
print("          EMNIST INFERENCE BENCHMARK COMPARISON TABLE")
print("=" * 70)
display(df_summary)

          EMNIST INFERENCE BENCHMARK COMPARISON TABLE


,Framework,Latency B=1 (ms),Latency B=64 (ms),Throughput (img/sec),Primary Advantage
4,ONNX Runtime,0.280 ms,3.926 ms,"16,301 items/s",Graph-optimized CPU inference
3,JAX / Flax (JIT),0.539 ms,0.529 ms,"120,982 items/s",XLA-compiled functional speed
1,PyTorch Lightning,0.934 ms,26.672 ms,"2,399 items/s",Clean production training code
0,PyTorch,1.214 ms,28.187 ms,"2,270 items/s",Ecosystem & dynamic graph
2,TensorFlow / Keras,15.491 ms,7.024 ms,"9,111 items/s",Beginner-friendly API
